# RAG (Retrieval-Augmented Generation)

Ce notebook explique pas à pas comment construire un petit système RAG avec LangChain.

**Document utilisé :** Règlement intérieur

---


## Imports et Configuration

On importe toutes les bibliothèques nécessaires et on charge les variables d'environnement.


In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)  # charge GROQ_API_KEY depuis .env

# Loaders - pour charger différents types de documents
from langchain_community.document_loaders import TextLoader

# Splitters - pour découper les documents en morceaux
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector Store - pour stocker et rechercher les embeddings
from langchain_community.vectorstores import FAISS

# Embeddings - pour convertir le texte en vecteurs
from langchain_huggingface import HuggingFaceEmbeddings

# LLM et chaînes
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\seb\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Chargement du Document

On charge notre document source. C'est la **base de connaissances** du RAG.


In [2]:
# Charger le document texte
loader = TextLoader("../data/reglement_interieur_enhanced.txt", encoding="utf-8")
docs = loader.load()

print(f"Nombre de documents : {len(docs)}")
print(f"Taille du document : {len(docs[0].page_content)} caractères")


Nombre de documents : 1
Taille du document : 14746 caractères


---
## Découpage en Chunks

On découpe le document en **petits morceaux (chunks)** pour :
1. Pouvoir les convertir en embeddings (limités en taille)
2. Récupérer seulement les parties pertinentes

```
Document complet
              ↓
    RecursiveCharacterTextSplitter
              ↓
┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐
│ Chunk 1  │ │ Chunk 2  │ │ Chunk 3  │ │ Chunk 4  │
│ 1000 car │ │ 1000 car │ │ 1000 car │ │ 1000 car │
└──────────┘ └──────────┘ └──────────┘ └──────────┘
      ↑__overlap__↑
```


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # Taille optimale pour un document juridique (articles complets)
    chunk_overlap=200,   # Chevauchement entre chunks (environ 20%)
    separators=["\n\n", "\n", ".", " "]  # Priorité de découpe
)

splits = splitter.split_documents(docs)

print(f"Nombre de chunks créés : {len(splits)}")
print(f"\nExemple du chunk 3 :")
print("=" * 50)
print(splits[3].page_content)


Nombre de chunks créés : 19

Exemple du chunk 3 :
PARTIE 2 – DROITS ET OBLIGATIONS (VERSION ENRICHIE)


------------------------------------------------------------
ARTICLE 2.6 – ANIMAUX DOMESTIQUES
------------------------------------------------------------

Mots-clés : chien, chat, animal, compagnie, perroquet, laisse

Questions courantes :
- Peut-on avoir des animaux de compagnie ?
- Les chiens sont-ils autorisés ?
- Mon chat peut-il sortir dans les communs ?

RÈGLES D'AUTORISATION :

CHIENS :
• Statut : TOLÉRÉS si petite taille uniquement
• Condition OBLIGATOIRE : Tenir en laisse en permanence
• INTERDIT : Laisser errer dans parties communes

CHATS :
• Statut : Autorisés dans les appartements
• INTERDIT : Laisser errer dans parties communes (couloirs, escaliers)

PERROQUETS :
• Statut : FORMELLEMENT INTERDITS
• Raison : Animaux criards perturbant la tranquillité


---
## Création des Embeddings et du VectorStore

Chaque chunk est converti en **vecteur (embedding)** puis stocké dans le **vectorstore**.

```
"Article Machin chose..." → Embedding Model → [0.12, -0.45, 0.78, ...]
                                                    ↓
                                              VectorStore
                                           (stockage + recherche)
```


In [4]:
# Modèle d'embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# On crée le vectorstore à partir des chunks (ici on utilise FAISS)
vectorstore = FAISS.from_documents(splits, embeddings)

# On crée le retriever (qui permet de récupérer les k documents les plus proches de la question de l'utilisateur)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Nombre de vecteurs indexés : {vectorstore.index.ntotal}")


Nombre de vecteurs indexés : 19


### Test du Retriever

Testons la recherche sémantique pour voir si le retriever trouve les bons documents :


In [5]:
question_test = "Les perroquets sont-ils autorisés ?"
docs_trouves = retriever.invoke(question_test)

print(f"Question : {question_test}")
print(f"{len(docs_trouves)} documents trouvés :\n")

for i, doc in enumerate(docs_trouves, 1):
    print(f"--- Document {i} ---")
    print(doc.page_content[:300])
    print()


Question : Les perroquets sont-ils autorisés ?
3 documents trouvés :

--- Document 1 ---
PERROQUETS :
• Statut : FORMELLEMENT INTERDITS
• Raison : Animaux criards perturbant la tranquillité

AUTRES ANIMAUX CRIARDS :
• Statut : INTERDITS

RESPONSABILITÉ PROPRIÉTAIRE :
• Tous dégâts, dégradations ou préjudices causés par les animaux
• Restent entièrement à la charge de leurs propriétaires

--- Document 2 ---
Q: Quels animaux sont interdits ?
R: Les perroquets et autres animaux criards sont formellement interdits selon l'Article 2.6.

Q: Puis-je faire des travaux dans mon appartement ?
R: Oui, vous pouvez modifier la distribution intérieure sous votre responsabilité selon l'Article 2.1. Les gros travaux 

--- Document 3 ---
TEXTE ORIGINAL :
"Les appartements ou locaux ne pourront être occupés que bourgeoisement ; toutefois, 
l'exercice des professions libérales est toléré dans tous les appartements."

Référence : Article 2.2 du règlement intérieur


---------------------------------------

---
## Création du Prompt

On crée un **template de prompt** qui combine le contexte (documents) et la question :


In [6]:
template = """Tu es un assistant juridique spécialisé dans les règlements de copropriété.

Réponds UNIQUEMENT à partir du contexte fourni ci-dessous.
Si tu ne trouves pas l'information, dis-le honnêtement.
Cite les articles pertinents quand c'est possible.

Contexte :
{context}

Question : {question}

Réponse (en français, claire et précise) :"""

prompt = ChatPromptTemplate.from_template(template)

---
## Configuration du LLM


In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3

---
## Chaîne RAG (LCEL)

On assemble tous les composants en une **chaîne LCEL** :

```
Question → {"context": retriever|format_docs, "question": passthrough}
                                    ↓
                                 prompt
                                    ↓
                                   llm
                                    ↓
                            StrOutputParser()
                                    ↓
                              Réponse finale
```


In [8]:
def format_docs(docs):
    """Formate les documents récupérés en une seule chaîne de texte"""
    return "\n\n".join(doc.page_content for doc in docs)


# LA CHAÎNE RAG COMPLÈTE
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

---
## Test du RAG !

Testons notre système avec des questions sur le règlement de copropriété


In [9]:
# Posez votre question ici
question = "Puis adopter un perroquet dans mon appartement ?"

reponse = rag_chain.invoke(question)
print(reponse)


**Non.**  
L’adoption d’un perroquet dans votre appartement est **interdite** par le règlement de copropriété.

- **Article 2.6** du règlement intérieur stipule : « Les perroquets et autres animaux criards sont formellement interdits. »  
- Cette interdiction s’applique à tout propriétaire ou occupant, qu’il soit résident ou tiers.  

Même si les chiens de petite taille sont tolérés sous certaines conditions, aucun dispositif d’exception n’est prévu pour les perroquets. En conséquence, la présence d’un tel animal constituerait une violation du règlement et pourrait entraîner des sanctions (mise en demeure, amendes, voire procédure judiciaire) ainsi que la responsabilité financière du propriétaire pour tout dommage éventuel, conformément à la clause de responsabilité prévue dans le même article.  

**Conclusion :** Vous ne pouvez pas adopter de perroquet dans votre appartement tant que le règlement de copropriété reste en vigueur.


---
## Résumé : Les Étapes du RAG

| Étape | Composant | Rôle |
|-------|-----------|------|
| 1 | **Loader** | Charge le document source |
| 2 | **Splitter** | Découpe en chunks |
| 3 | **Embeddings** | Convertit texte → vecteurs |
| 4 | **VectorStore** | Stocke et indexe les vecteurs |
| 5 | **Retriever** | Recherche les docs similaires |
| 6 | **Prompt** | Template avec context + question |
| 7 | **LLM** | Génère la réponse |
| 8 | **Parser** | Extrait le texte final |
